# ASD-XMF X1 Datasets Benchmarks
Stage 1 datasets D2-D4, then the audio, video and text sector comparisons (A1, V1, T1).

Each section is one former notebook, run as its own process. A section waits for the ones it needs; a time guard stops before Kaggle's 12-hour limit and the next run continues where this one stopped.

| section | source | needs |
|---|---|---|
| `d2` | `d2_audio_builder.py` | - |
| `d3` | `d3_video_builder.py` | - |
| `d4` | `d4_text_builder.py` | - |
| `v1` | `v1_video_benchmark.py` | d3 |
| `t1` | `t1_text_benchmark.py` | d4 |
| `a1` | `a1_audio_benchmark.py` | d2 |

In [ ]:
RUN_PROFILE = "full"   # "full" on GPU; "preflight" = quick CPU smoke test (tools/push.py)
PREFLIGHT = RUN_PROFILE == "preflight"
import os, sys, glob, json, time, warnings
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def _find_lib():
    for c in ("/kaggle/input/asd-xmf-lib", "/kaggle/input/datasets/shazedaoni/asd-xmf-lib"):
        if os.path.exists(os.path.join(c, "asd_utils.py")):
            return c
    for depth in range(1, 6):
        hits = glob.glob("/kaggle/input/" + "*/" * depth + "asd_utils.py")
        if hits:
            return os.path.dirname(hits[0])
    return os.path.abspath(os.path.join(os.getcwd(), "..", "..", "asd_lib"))  # local run

def _require_internet():
    import socket
    try:
        socket.create_connection(("pypi.org", 443), timeout=10).close()
    except OSError:
        raise SystemExit("NO INTERNET in this Kaggle session: verify the account's phone number (Kaggle > Settings) "
                         "and keep 'Internet on'. Stopping so no step runs with missing models or features.")
_require_internet()

LIB = _find_lib()
sys.path.insert(0, LIB)
import asd_utils as U
print("asd_lib:", LIB, "| profile:", RUN_PROFILE)
ENV = U.env_report()
open(os.path.join(str(U.WORK), "_run_profile_" + RUN_PROFILE + ".txt"), "w").write(time.strftime("%Y-%m-%d %H:%M:%S"))  # queue marker

In [ ]:
import asd_runner as R
R.setup('shazedaoni/asd-xmf-x1-datasets-benchmarks', RUN_PROFILE, [{"name": "d2", "after": [], "rev": 1}, {"name": "d3", "after": [], "rev": 1}, {"name": "d4", "after": [], "rev": 1}, {"name": "v1", "after": ["d3"], "rev": 2}, {"name": "t1", "after": ["d4"], "rev": 2}, {"name": "a1", "after": ["d2"], "rev": 2}], hours=11.0, rev=2)

## Section `d2`: D2 - XMF-Audio: unified voice + cry dataset with cached features

In [ ]:
%%writefile {R.SEC_DIR}/d2.py
# # D2 - XMF-Audio: unified voice + cry dataset with cached features
# | task | source | label | grouping for CV |
# |---|---|---|---|
# | `vocal_affect` | ReCANVo (8 minimally verbal autistic people) | caregiver affect label (classes with >= 100 clips) | speaker |
# | `cry_reason` | Donate-a-Cry | belly pain / burping / discomfort / hungry / tired | infant ID |
# | `cry_detect` | infant-cry dataset | cry vs. non-cry | none available (stated as a limitation) |
#
# Exact-duplicate audio (MD5) is removed across all sources.  Each file is decoded once (16 kHz mono)
# and cached as: 64-band log-mel, MFCC-40 statistics (proposal), eGeMAPS-v02 (openSMILE),
# wav2vec2-base layer means, AST and PANNs-CNN14 embeddings.
# 

import hashlib, re
import numpy as np, pandas as pd
from pathlib import Path
import asd_audio as A
for pkg in ("opensmile", "panns-inference"):
    U.pip_install(pkg)
OUT = U.WORK / "xmf_audio"; OUT.mkdir(parents=True, exist_ok=True)


items = []
meta = U.find_path("dataset_file_directory.csv"); rc_root = meta.parent
wav = {p.name: p for p in rc_root.rglob("*.wav")}
rc = pd.read_csv(meta)
keep = rc.Label.value_counts(); keep = keep[keep >= 100].index
for r in rc[rc.Label.isin(keep)].itertuples():
    if r.Filename in wav:
        items.append({"path": str(wav[r.Filename]), "task": "vocal_affect", "label": r.Label, "group": r.Participant, "source": "recanvo"})
cry_dir = U.find_path("donateacry_corpus", want="dir")
for cdir in (p for p in cry_dir.iterdir() if p.is_dir()):
    for f in cdir.glob("*.wav"):
        m = re.match(r"^([0-9A-Fa-f-]{36})", f.name)
        items.append({"path": str(f), "task": "cry_reason", "label": cdir.name, "group": m.group(1).lower() if m else f.stem[:8], "source": "donateacry"})
det = U.find_path("cry", want="dir")
for cdir in (p for p in det.parent.iterdir() if p.is_dir()):
    for f in cdir.glob("*.wav"):
        items.append({"path": str(f), "task": "cry_detect", "label": "cry" if cdir.name.lower() == "cry" else "non_cry",
                      "group": f.stem, "source": f"infantcry/{cdir.name}"})
items = pd.DataFrame(items)
items["md5"] = [hashlib.md5(Path(p).read_bytes()).hexdigest() for p in items.path]
n0 = len(items)
items = items.drop_duplicates("md5").reset_index(drop=True)
print(f"removed {n0 - len(items)} exact duplicates")
print(items.groupby(["task", "label"]).size())
if PREFLIGHT:
    items = items.groupby(["task", "label"]).head(4).reset_index(drop=True)


items["relpath"] = items.path
with U.Timer("features"):
    A.extract_all(items, "relpath", Path("/"), OUT, "audio", use_cnn14=not PREFLIGHT, max_s=10)
idx = pd.read_csv(OUT / "audio_index.csv")
print(idx.groupby("task")[["dur_s", "voiced_frac"]].mean())
(OUT / "DATASHEET.md").write_text(f"""# XMF-Audio
{len(idx)} clips after MD5 de-duplication. Tasks: {idx.task.value_counts().to_dict()}.
Sources: ReCANVo (Zenodo 5786860, CC BY 3.0 US), Donate-a-Cry (warcoder/infant-cry-audio-corpus), infant-cry dataset (sanmithasadhish/infant-cry-dataset).
Features per clip: log-mel (audio_mel/), MFCC-40 stats, eGeMAPS-v02, wav2vec2-base layer means, AST, CNN14.
Not an autism-diagnosis dataset: used to pre-train the voice and cry branches.
""")


In [ ]:
R.run('d2')

## Section `d3`: D3 - XMF-Video: behaviour sequences (MMASD 2D skeleton)

In [ ]:
%%writefile {R.SEC_DIR}/d3.py
# # D3 - XMF-Video: behaviour sequences (MMASD 2D skeleton)
# MMASD (IRB-approved, parental consent; 32 autistic children, therapy activities) ships privacy-preserving
# OpenPose skeletons instead of raw video.  Plan 4.2: pre-train the motion/behaviour branch.
# Each clip -> 64 frames x 25 joints x (x, y, confidence), centred on the neck and scaled by torso
# length, with a per-frame validity mask.  The subject ID is kept so that every split is by child.
# The eye-tracking (gaze) data are used in NB13 with the same child-wise protocol.
# 

import json as _json, re
import numpy as np, pandas as pd
from pathlib import Path
import multiprocessing as mp
OUT = U.WORK / "xmf_video"; OUT.mkdir(parents=True, exist_ok=True)
T = 64
root = U.find_path("2D_openpose", want="dir")
clips = sorted({p.parent for p in root.rglob("*_keypoints.json")})
print("clips:", len(clips))
if PREFLIGHT:
    clips = clips[:60]

def load_clip(d):
    d = Path(d)
    frames = sorted(d.glob("*_keypoints.json"))
    seq = []
    for f in frames:
        try:
            ppl = _json.loads(f.read_text()).get("people", [])
        except Exception:
            ppl = []
        if ppl:  # most confident person in the frame
            k = max((np.array(p["pose_keypoints_2d"], np.float32).reshape(-1, 3) for p in ppl), key=lambda a: a[:, 2].sum())
            seq.append(k[:25])
        else:
            seq.append(np.zeros((25, 3), np.float32))
    seq = np.stack(seq) if seq else np.zeros((1, 25, 3), np.float32)
    valid = seq[:, :, 2].sum(1) > 0
    neck, hip = seq[:, 1, :2], seq[:, 8, :2]
    scale = np.linalg.norm(neck - hip, axis=1)
    scale = np.where(scale > 1, scale, np.nanmedian(scale[scale > 1]) if (scale > 1).any() else 1.0)
    xy = (seq[:, :, :2] - neck[:, None]) / scale[:, None, None]
    feat = np.concatenate([xy, seq[:, :, 2:3]], -1) * valid[:, None, None]
    ids = np.linspace(0, len(feat) - 1, T).round().astype(int)
    return feat[ids].reshape(T, -1).astype(np.float32), valid[ids], len(frames)

with mp.get_context("fork").Pool(os.cpu_count()) as pool:  # json parsing only: fork is safe here
    out = pool.map(load_clip, [str(c) for c in clips], chunksize=8)
X = np.stack([o[0] for o in out]); Mk = np.stack([o[1] for o in out])
meta = []
for c, o in zip(clips, out):
    m = re.match(r"^([a-z]+)_(\d+)_", c.name)
    meta.append({"clip": c.name, "activity": c.parent.name, "subject": m.group(2) if m else c.name.split("_")[1], "n_frames": o[2]})
meta = pd.DataFrame(meta)
keep = Mk.sum(1) >= T // 4
X, Mk, meta = X[keep], Mk[keep], meta[keep].reset_index(drop=True)
np.save(OUT / "skeleton.npy", X); np.save(OUT / "mask.npy", Mk)
meta.to_csv(OUT / "manifest.csv", index=False)
print(meta.activity.value_counts(), "\nsubjects:", meta.subject.nunique(), "| clips kept:", len(meta))
(OUT / "DATASHEET.md").write_text(f"""# XMF-Video
{len(meta)} MMASD skeleton clips, {meta.subject.nunique()} children, {meta.activity.nunique()} activities.
Tensor: {T} frames x 25 joints x (x, y, conf), neck-centred, torso-scaled; mask = frame has a detected person.
Source: MMASD (arXiv 2306.08243) via gursewakbfcet/mmasd-final-dataset. Autistic children only: activity labels, not diagnosis.
""")


In [ ]:
R.run('d3')

## Section `d4`: D4 - XMF-Text: free text + questionnaire items

In [ ]:
%%writefile {R.SEC_DIR}/d4.py
# # D4 - XMF-Text: free text + questionnaire items
# * `text.csv` - TASD: statements about a child's behaviour, labelled ASD / non-ASD (plan 5.1: free-text parent
#   concerns; BanglaBERT once Bangla text is collected).
# * `quest.csv` - Q-CHAT-10 toddlers and AQ-10 screening answers.  Plan 4.2 warns that their label is derived
#   from the item scores, so they are used only for a **leakage audit**, never as a headline result.
# Exact duplicate rows are removed.
# 

import numpy as np, pandas as pd
OUT = U.WORK / "xmf_text"; OUT.mkdir(parents=True, exist_ok=True)
t = pd.read_csv(U.find_path("Dataset-v2.csv"))
text = pd.DataFrame({"text": t.Text.astype(str).str.strip(), "label": (t.ASD.str.strip() == "ASD").astype(int),
                     "behaviour": t.Class, "source": "tasd"}).drop_duplicates("text").reset_index(drop=True)
text.to_csv(OUT / "text.csv", index=False)
q1 = pd.read_csv(U.find_path("Toddler Autism dataset July 2018.csv"))
lab = [c for c in q1.columns if c.strip().startswith("Class")][0]
quest = [pd.DataFrame({**{f"A{i}": q1[f"A{i}"] for i in range(1, 11)}, "age": q1.Age_Mons / 12.0, "sex": (q1.Sex == "m").astype(int),
                       "label": (q1[lab].str.strip().str.lower() == "yes").astype(int), "source": "qchat10_toddler"})]
comb = U.find_path("Autism_Screening_Data_Combined.csv")
if comb is not None:
    q2 = pd.read_csv(comb)
    quest.append(pd.DataFrame({**{f"A{i}": q2[f"A{i}"] for i in range(1, 11)}, "age": q2.Age, "sex": (q2.Sex == "m").astype(int),
                               "label": (q2.Class.str.strip().str.upper() == "YES").astype(int), "source": "aq10_combined"}))
quest = pd.concat(quest, ignore_index=True).drop_duplicates().reset_index(drop=True)
quest.to_csv(OUT / "quest.csv", index=False)
print(text.label.value_counts().to_dict(), "| quest:", quest.groupby("source").label.agg(["size", "mean"]))
(OUT / "DATASHEET.md").write_text(f"""# XMF-Text
text.csv: {len(text)} TASD statements (ASD {int(text.label.sum())}, non-ASD {int((1 - text.label).sum())}).
quest.csv: {len(quest)} questionnaire rows ({quest.source.value_counts().to_dict()}); label derived from item sum -> leakage audit only.
""")


In [ ]:
R.run('d4')

## Section `v1`: V1 - Video sector: proposed XMF temporal encoder vs baselines (MMASD behaviour activities)

In [ ]:
%%writefile {R.SEC_DIR}/v1.py
# # V1 - Video sector: proposed XMF temporal encoder vs baselines (MMASD behaviour activities)
# **Proposed (plan 5.1):** XMF-Temporal = per-frame projection -> 2-layer transformer -> attention pooling
# (the same temporal encoder the face-video branch uses).  **Baselines:** pooled-statistics MLP, 1D-CNN,
# BiLSTM, CNN-LSTM (proposal).  5-fold StratifiedGroupKFold **by child** (no child in both train and test),
# an inner child-wise validation split for early stopping, identical folds for every model.
# 

import numpy as np, pandas as pd
import asd_bench as B
SRC = U.find_path("skeleton.npy").parent
OUT = U.WORK / "v1"; OUT.mkdir(parents=True, exist_ok=True)
X = np.load(SRC / "skeleton.npy"); Mk = np.load(SRC / "mask.npy")
meta = pd.read_csv(SRC / "manifest.csv", dtype={"subject": str})
classes = sorted(meta.activity.unique()); y = meta.activity.map({c: i for i, c in enumerate(classes)}).values
print(X.shape, len(classes), "activities,", meta.subject.nunique(), "children")
with U.Timer("video benchmark"):
    rows = B.seq_cv(X, Mk, y, meta.subject.values, len(classes), list(B.SEQ_MODELS), OUT, "mmasd_activity",
                    epochs=3 if PREFLIGHT else 150, folds=3 if PREFLIGHT else 5, max_steps=5 if PREFLIGHT else None)
tab = B.comparison_table(rows)
tab["majority_baseline"] = pd.Series(y).value_counts(normalize=True).iloc[0]
print(tab.to_markdown(index=False, floatfmt=".3f"))
tab.to_csv(OUT / "video_comparison.csv", index=False)


In [ ]:
R.run('v1')

## Section `t1`: T1 - Text sector: proposed XMF-Text vs baselines (ASD vs non-ASD statements) + questionnaire leakage audit

In [ ]:
%%writefile {R.SEC_DIR}/t1.py
# # T1 - Text sector: proposed XMF-Text vs baselines (ASD vs non-ASD statements) + questionnaire leakage audit
# **Proposed (plan 5.1 text branch):** XMF-Text v2 = two branches fused late:
# * semantic: multilingual XLM-RoBERTa-base encoder (ready for Bangla text) -> 256-d token projection ->
#   attention pooling -> classifier
# * lexical: word 1-2-gram TF-IDF + logistic regression (parents' word choice carries much of the signal)
#
# Both branches are trained on the same inner training split; the fusion weight w in
# p = w * p_semantic + (1 - w) * p_lexical is chosen on the inner validation split only, never on the test fold.
# **Baselines:** TF-IDF + LR, TF-IDF + SVM, BiLSTM, DistilBERT, BERT-base, RoBERTa-base.
# Repeated stratified 5-fold CV (same folds for all), inner validation split for early stopping,
# train vs validation accuracy reported for every model.
# 

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
import asd_models as M, asd_bench as B
SRC = U.find_path("text.csv").parent
OUT = U.WORK / "t1"; OUT.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(SRC / "text.csv"); X, y = df.text.values, df.label.values
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=0).split(X, y))
if PREFLIGHT:
    FOLDS = FOLDS[:2]
dev = M.device()

class AttnPoolCls(nn.Module):
    def __init__(self, enc, hid, proposed):
        super().__init__()
        self.enc, self.proposed = enc, proposed
        self.proj = nn.Sequential(nn.Linear(hid, 256), nn.GELU(), nn.LayerNorm(256)) if proposed else nn.Identity()
        d = 256 if proposed else hid
        self.att = nn.Linear(d, 1) if proposed else None
        self.fc = nn.Sequential(nn.Dropout(0.2), nn.Linear(d, 2))

    def forward(self, ids, am):
        h = self.enc(input_ids=ids, attention_mask=am).last_hidden_state
        if self.proposed:
            h = self.proj(h)
            a = self.att(h).squeeze(-1).masked_fill(am == 0, -1e4).softmax(-1)
            return self.fc((a.unsqueeze(-1) * h).sum(1))
        return self.fc(h[:, 0])

def lexical_branch(fit, idx_list):
    """TF-IDF + LR fitted on `fit`; returns P(ASD) for each index array in idx_list."""
    v = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True); A = v.fit_transform(X[fit])
    c = LogisticRegression(C=4.0, max_iter=3000).fit(A, y[fit])
    return [c.predict_proba(v.transform(X[i]))[:, 1] for i in idx_list]

def run_transformer(name, ckpt, proposed=False):
    from transformers import AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(ckpt)
    enc = tok(list(X), truncation=True, max_length=128, padding="max_length", return_tensors="pt")
    ids, am = enc["input_ids"].to(dev), enc["attention_mask"].to(dev)
    pred, trs, vas = np.full(len(y), -1), [], []
    fusion_w = []
    for f, (tr, te) in enumerate(FOLDS):
        U.seed_everything(f)
        rng = np.random.default_rng(f); va = rng.choice(tr, len(tr) // 8, replace=False); fit = np.setdiff1d(tr, va)
        base = AutoModel.from_pretrained(ckpt)
        model = AttnPoolCls(base, base.config.hidden_size, proposed).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        scaler = M.make_scaler(); best, best_state, best_ep = -1, None, 0
        Y = torch.tensor(y, device=dev)
        pr = lambda idx: torch.cat([model(ids[idx[s:s + 64]], am[idx[s:s + 64]]).argmax(1) for s in range(0, len(idx), 64)]).cpu().numpy()
        for ep in range(1 if PREFLIGHT else 12):
            model.train()
            for s in range(0, len(fit), 16):
                i = torch.tensor(fit[rng.permutation(len(fit))][s:s + 16], device=dev)
                with M.amp_ctx():
                    loss = F.cross_entropy(model(ids[i], am[i]).float(), Y[i], label_smoothing=0.1)
                opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            model.eval()
            with torch.no_grad(), M.amp_ctx():
                acc_va = (pr(va) == y[va]).mean()
            if acc_va > best:
                best, best_ep, best_state = acc_va, ep, {k: v.detach().clone() for k, v in model.state_dict().items()}
            elif ep - best_ep >= 3:
                break
        model.load_state_dict(best_state); model.eval()
        if proposed:
            prob = lambda idx: torch.cat([model(ids[idx[s:s + 64]], am[idx[s:s + 64]]).float().softmax(1)[:, 1]
                                          for s in range(0, len(idx), 64)]).cpu().numpy()
            with torch.no_grad(), M.amp_ctx():
                ps_va, ps_te, ps_fit = prob(va), prob(te), prob(fit)
            pl_va, pl_te, pl_fit = lexical_branch(fit, [va, te, fit])
            grid = np.round(np.linspace(0, 1, 11), 2)   # fusion weight chosen on validation only (ties -> 0.5)
            accs = [((w * ps_va + (1 - w) * pl_va > 0.5) == y[va]).mean() for w in grid]
            w = max(grid, key=lambda g: (accs[list(grid).index(g)], -abs(g - 0.5)))
            fusion_w.append(float(w))
            pred[te] = (w * ps_te + (1 - w) * pl_te > 0.5).astype(int)
            trs.append(((w * ps_fit + (1 - w) * pl_fit > 0.5) == y[fit]).mean()); vas.append(max(accs))
        else:
            with torch.no_grad(), M.amp_ctx():
                pred[te] = pr(te); trs.append((pr(fit) == y[fit]).mean()); vas.append(best)
        del model, base; torch.cuda.empty_cache()
    if proposed:
        print(f"{name}: fusion weight on the semantic branch per fold (chosen on validation): {fusion_w}")
    return pred, trs, vas

def run_tfidf(clf):
    pred, trs = np.full(len(y), -1), []
    for tr, te in FOLDS:
        v = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True); A = v.fit_transform(X[tr])
        c = clf().fit(A, y[tr]); pred[te] = c.predict(v.transform(X[te])); trs.append((c.predict(A) == y[tr]).mean())
    return pred, trs, [np.nan] * len(trs)

class BiLSTMText(nn.Module):
    def __init__(self, vocab, d=128):
        super().__init__()
        self.emb = nn.Embedding(vocab, d, padding_idx=0); self.lstm = nn.LSTM(d, d, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(nn.Dropout(0.4), nn.Linear(2 * d, 2))
    def forward(self, ids):
        h, _ = self.lstm(self.emb(ids)); m = (ids > 0).float().unsqueeze(-1)
        return self.fc((h * m).sum(1) / m.sum(1).clamp_min(1))

def run_bilstm():
    from collections import Counter
    pred, trs, vas = np.full(len(y), -1), [], []
    for f, (tr, te) in enumerate(FOLDS):
        rng = np.random.default_rng(f); va = rng.choice(tr, len(tr) // 8, replace=False); fit = np.setdiff1d(tr, va)
        cnt = Counter(w for s in X[fit] for w in s.lower().split()); vocab = {w: i + 2 for i, (w, _) in enumerate(cnt.most_common(20000))}
        enc = lambda s: ([vocab.get(w, 1) for w in s.lower().split()][:64] + [0] * 64)[:64]
        I = torch.tensor([enc(s) for s in X], device=dev); Y = torch.tensor(y, device=dev)
        net = BiLSTMText(len(vocab) + 2).to(dev); opt = torch.optim.AdamW(net.parameters(), lr=2e-3, weight_decay=0.05)
        best, best_state, best_ep = -1, None, 0
        for ep in range(2 if PREFLIGHT else 60):
            net.train()
            for s in range(0, len(fit), 32):
                i = torch.tensor(fit[rng.permutation(len(fit))][s:s + 32], device=dev)
                loss = F.cross_entropy(net(I[i]), Y[i], label_smoothing=0.1); opt.zero_grad(); loss.backward(); opt.step()
            net.eval()
            with torch.no_grad():
                a = (net(I[va]).argmax(1).cpu().numpy() == y[va]).mean()
            if a > best:
                best, best_ep, best_state = a, ep, {k: v.clone() for k, v in net.state_dict().items()}
            elif ep - best_ep >= 8:
                break
        net.load_state_dict(best_state); net.eval()
        with torch.no_grad():
            pred[te] = net(I[te]).argmax(1).cpu().numpy(); trs.append((net(I[fit]).argmax(1).cpu().numpy() == y[fit]).mean()); vas.append(best)
    return pred, trs, vas


MODELS = {"tfidf_lr": lambda: run_tfidf(lambda: LogisticRegression(C=4.0, max_iter=3000)),
          "tfidf_svm": lambda: run_tfidf(lambda: LinearSVC(C=0.5)),
          "bilstm": run_bilstm,
          "distilbert": lambda: run_transformer("distilbert", "distilbert-base-uncased"),
          "bert_base": lambda: run_transformer("bert_base", "bert-base-uncased"),
          "roberta_base": lambda: run_transformer("roberta_base", "roberta-base"),
          "xmf_text": lambda: run_transformer("xmf_text", "xlm-roberta-base", proposed=True)}
if PREFLIGHT:
    MODELS = {k: MODELS[k] for k in ("tfidf_lr", "bilstm", "xmf_text")}
rows = []
for name, fn in MODELS.items():
    with U.Timer(name):
        p, trs, vas = fn()
    m = B.cls_metrics(y[p >= 0], p[p >= 0], 2)
    lo, hi = B.boot_ci(y[p >= 0], p[p >= 0], lambda a, b: float((a == b).mean()))
    rows.append({"task": "tasd_asd_vs_nonasd", "model": name, "proposed": name == "xmf_text", "test_acc": m["acc"], "ci_lo": lo, "ci_hi": hi,
                 "test_macro_f1": m["macro_f1"], "train_acc": float(np.mean(trs)), "val_acc": float(np.nanmean(vas)) if np.isfinite(vas).any() else np.nan,
                 "gap": float(np.mean(trs) - m["acc"])})
    B._save(OUT, f"tasd__{name}", rows[-1] | {"cm": m["cm"]}, [])
    print(rows[-1], flush=True)
tab = B.comparison_table(pd.DataFrame(rows))
print(tab.to_markdown(index=False, floatfmt=".3f"))
tab.to_csv(OUT / "text_comparison.csv", index=False)


# ## Questionnaire leakage audit (plan 4.2: label is derived from the item scores)
# 

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict
q = pd.read_csv(SRC / "quest.csv")
audit = []
for src, g in q.groupby("source"):
    items = g[[f"A{i}" for i in range(1, 11)]].values
    score = items.sum(1)
    best_thr = max(range(11), key=lambda t: ((score > t) == g.label.values).mean())
    rule_acc = ((score > best_thr) == g.label.values).mean()
    rf = cross_val_predict(RandomForestClassifier(300, random_state=0), items, g.label.values, cv=5)
    audit.append({"source": src, "n": len(g), "rule": f"sum(A1..A10) > {best_thr}", "rule_accuracy": rule_acc,
                  "random_forest_cv_accuracy": (rf == g.label.values).mean()})
audit = pd.DataFrame(audit)
print(audit.to_markdown(index=False, floatfmt=".4f"))
print("A rule accuracy of ~1.00 means the label is a deterministic function of the items: these sets cannot "
      "support an ASD-screening claim (plan 4.2) and are reported only as a leakage check.")
audit.to_csv(OUT / "questionnaire_leakage_audit.csv", index=False)


In [ ]:
R.run('t1')

## Section `a1`: A1 - Audio sector: proposed XMF-Audio vs baselines (voice and cry)

In [ ]:
%%writefile {R.SEC_DIR}/a1.py
# # A1 - Audio sector: proposed XMF-Audio vs baselines (voice and cry)
# **Proposed (plan 5.1 voice/cry branch):** XMF-Audio v2 = gated fusion of five feature blocks
# (wav2vec2-base mid-layer mean, eGeMAPS prosody, MFCC-40 statistics, AST and CNN14 embeddings): each block ->
# 128-d projection, a learned softmax gate weights the blocks, block dropout during training, fully
# class-balanced loss, and the probabilities of 3 independently seeded heads are averaged.
# Imbalanced tasks are ranked by macro-F1 (accuracy mostly rewards the majority class there).
# **Baselines:** MFCC + SVM, eGeMAPS + LR, wav2vec2 + LR, AST + LR, CNN14 + LR, log-mel EfficientNet-B0 CNN,
# log-mel CNN-LSTM (proposal).
#
# Cross-validation never mixes a speaker / infant between train and test: leave-one-speaker-out for
# `vocal_affect`, 5-fold group CV by infant for `cry_reason`, stratified 5-fold for `cry_detect`.
# Every model uses the same folds.  Train vs validation accuracy is reported (over/underfitting).
# 

import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
import asd_models as M, asd_bench as B
SRC = U.find_path("audio_index.csv").parent
OUT = U.WORK / "a1"; OUT.mkdir(parents=True, exist_ok=True)
idx = pd.read_csv(SRC / "audio_index.csv")
def feat(name):
    p = SRC / f"audio_{name}.npy"
    if not p.exists():
        return None
    X = np.load(p).astype(np.float32)
    return np.nan_to_num(X[:, 4:10].mean(1) if name == "w2v" else X)
FEATS = {k: feat(k) for k in ("mfcc", "egemaps", "w2v", "ast", "cnn14")}
FEATS = {k: v for k, v in FEATS.items() if v is not None}
print({k: v.shape for k, v in FEATS.items()})

def folds(task_rows):
    d = idx.loc[task_rows]
    y, g = d.label.values, d.group.values
    if d.task.iloc[0] == "vocal_affect":
        return [(task_rows[g != s], task_rows[g == s]) for s in sorted(set(g))]
    k = int(max(2, min(5, pd.Series(y).value_counts().min())))  # 5 folds; fewer only if a class is tiny
    if d.task.iloc[0] == "cry_reason":
        cv = StratifiedGroupKFold(k, shuffle=True, random_state=0)
        return [(task_rows[a], task_rows[b]) for a, b in cv.split(task_rows, y, g)]
    cv = StratifiedKFold(k, shuffle=True, random_state=0)
    return [(task_rows[a], task_rows[b]) for a, b in cv.split(task_rows, y)]


# ## Proposed XMF-Audio head
# 

class XMFAudio(nn.Module):
    def __init__(self, dims, n_cls, d=128, p_drop=0.3):
        super().__init__()
        self.proj = nn.ModuleList([nn.Sequential(nn.Linear(k, d), nn.GELU(), nn.LayerNorm(d), nn.Dropout(0.2)) for k in dims])
        self.gate = nn.Linear(d * len(dims), len(dims))
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(d, n_cls))
        self.p_drop = p_drop

    def forward(self, xs):
        H = torch.stack([p(x) for p, x in zip(self.proj, xs)], 1)        # B, blocks, d
        g = self.gate(H.flatten(1))
        if self.training and self.p_drop:                                 # block dropout
            drop = torch.rand_like(g) < self.p_drop
            drop[drop.all(1)] = False
            g = g.masked_fill(drop, -1e4)
        return self.head((g.softmax(-1).unsqueeze(-1) * H).sum(1))

def fit_xmf(blocks, y, tr, te, n_cls, seed=0, epochs=200):
    dev = M.device(); U.seed_everything(seed)
    rng = np.random.default_rng(seed)
    va = rng.choice(tr, max(2, len(tr) // 7), replace=False); fit = np.setdiff1d(tr, va)
    mus = [(b[fit].mean(0), b[fit].std(0) + 1e-5) for b in blocks]
    T = [torch.tensor((b - m) / s, dtype=torch.float32, device=dev) for b, (m, s) in zip(blocks, mus)]
    Y = torch.tensor(y, device=dev)
    w = torch.tensor(np.bincount(y[fit], minlength=n_cls).clip(1) ** -1.0, dtype=torch.float32, device=dev)  # fully balanced
    net = XMFAudio([b.shape[1] for b in blocks], n_cls).to(dev)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.05)
    best, best_state, best_ep = -1, None, 0
    for ep in range(epochs):
        net.train()
        for s in range(0, len(fit), 128):
            i = torch.tensor(fit[rng.permutation(len(fit))][s:s + 128], device=dev)
            loss = F.cross_entropy(net([t[i] for t in T]), Y[i], weight=w / w.mean(), label_smoothing=0.1)
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            pv = net([t[va] for t in T]).argmax(1).cpu().numpy()
        f1 = B.cls_metrics(y[va], pv, n_cls)["macro_f1"]
        if f1 > best:
            best, best_ep, best_state = f1, ep, {k: v.clone() for k, v in net.state_dict().items()}
        elif ep - best_ep >= 25:
            break
    net.load_state_dict(best_state); net.eval()
    with torch.no_grad():
        p = lambda ids: net([t[ids] for t in T]).softmax(1).cpu().numpy()
        return p(te), (p(fit).argmax(1) == y[fit]).mean(), (p(va).argmax(1) == y[va]).mean()

def fit_xmf_ensemble(blocks, y, tr, te, n_cls, seeds=(0, 1, 2), epochs=200):
    """XMF-Audio v2: average the class probabilities of independently seeded heads (own inner validation split each)."""
    outs = [fit_xmf(blocks, y, tr, te, n_cls, seed=s, epochs=epochs) for s in seeds]
    prob = np.mean([o[0] for o in outs], 0)
    return prob.argmax(1), float(np.mean([o[1] for o in outs])), float(np.mean([o[2] for o in outs]))


# ## Run every model on the same folds
# 

rows = []
for task, g in idx.groupby("task"):
    rows_t = g.index.to_numpy()
    classes = sorted(g.label.unique()); cmap = {c: i for i, c in enumerate(classes)}
    y = idx.label.map(cmap).fillna(-1).astype(int).values
    n_cls = len(classes)
    F_ = folds(rows_t)
    if PREFLIGHT:
        F_ = F_[:2]
    models = {f"{k}_svm" if k == "mfcc" else f"{k}_lr": k for k in FEATS}
    preds = {m: np.full(len(idx), -1) for m in list(models) + ["xmf_audio"]}
    tr_acc = {m: [] for m in preds}
    va_acc = {m: [] for m in preds}
    for tr, te in F_:
        for m, k in models.items():
            X = FEATS[k]
            clf = (make_pipeline(StandardScaler(), SVC(C=3.0, class_weight="balanced")) if m.endswith("_svm")
                   else make_pipeline(StandardScaler(), LogisticRegression(C=0.05, max_iter=3000, class_weight="balanced")))
            clf.fit(X[tr], y[tr])
            preds[m][te] = clf.predict(X[te]); tr_acc[m].append((clf.predict(X[tr]) == y[tr]).mean()); va_acc[m].append(np.nan)
        blocks = [FEATS[k] for k in ("w2v", "egemaps", "mfcc", "ast", "cnn14") if k in FEATS]
        pt, a_tr, a_va = fit_xmf_ensemble(blocks, y, tr, te, n_cls, seeds=(0,) if PREFLIGHT else (0, 1, 2),
                                          epochs=3 if PREFLIGHT else 200)
        preds["xmf_audio"][te] = pt; tr_acc["xmf_audio"].append(a_tr); va_acc["xmf_audio"].append(a_va)
    # log-mel CNN and CNN-LSTM on GPU, same folds
    lab = OUT / f"{task}_items.csv"; idx.assign(label_id=y).to_csv(lab, index=False)
    jobs = [{"name": f"{arch}_{task}_f{f}", "arch": arch, "index": str(lab), "mel_dir": str(SRC / "audio_mel"), "label_col": "label_id",
             "train_rows": tr.tolist(), "test_rows": te.tolist(), "fold": f, "out_dir": str(OUT), "seed": 0,
             "epochs": 1 if PREFLIGHT else 30, "batch": 64, "lr": 1e-3, "max_steps": 3 if PREFLIGHT else None, "crop_frames": 300}
            for arch in ("effb0", "cnn_lstm") for f, (tr, te) in enumerate(F_)]
    U.run_jobs_parallel("asd_audio", "melcnn_fold_task", jobs, log_dir=OUT / "logs")
    for arch, name in (("effb0", "logmel_cnn_effb0"), ("cnn_lstm", "logmel_cnn_lstm")):
        preds[name] = np.full(len(idx), -1); tr_acc[name] = []; va_acc[name] = []
        for f in range(len(F_)):
            d = pd.read_csv(OUT / "preds" / f"{arch}_{task}_f{f}.csv")
            pc = [c for c in d.columns if c.startswith("p_")]
            preds[name][d.row.values] = np.array([int(float(c[2:])) for c in pc])[d[pc].values.argmax(1)]
            tr_acc[name].append(d.train_acc.iloc[0]); va_acc[name].append(np.nan)
    for m, p in preds.items():
        sel = p[rows_t] >= 0
        mm = B.cls_metrics(y[rows_t][sel], p[rows_t][sel], n_cls)
        lo, hi = B.boot_ci(y[rows_t][sel], p[rows_t][sel], lambda a, b: float((a == b).mean()))
        rows.append({"task": task, "model": m, "proposed": m == "xmf_audio", "n": mm["n"], "test_acc": mm["acc"], "ci_lo": lo, "ci_hi": hi,
                     "test_macro_f1": mm["macro_f1"], "test_bal_acc": mm["bal_acc"], "train_acc": float(np.nanmean(tr_acc[m])),
                     "val_acc": float(np.nanmean(va_acc[m])) if np.isfinite(va_acc[m]).any() else np.nan,
                     "gap": float(np.nanmean(tr_acc[m]) - mm["acc"]), "majority_baseline": float(pd.Series(y[rows_t]).value_counts(normalize=True).iloc[0])})
        B._save(OUT, f"{task}__{m}", rows[-1] | {"cm": mm["cm"], "classes": classes}, [])


# ranked by macro-F1: cry reason and vocal affect are imbalanced, so accuracy mostly rewards the majority class
tab = B.comparison_table(pd.DataFrame(rows), metric="test_macro_f1")
print(tab.to_markdown(index=False, floatfmt=".3f"))
tab.to_csv(OUT / "audio_comparison.csv", index=False)


In [ ]:
R.run('a1')

In [ ]:
R.finish()